## Tool Creation

In [15]:
#load the API

import os 
from dotenv import find_dotenv, load_dotenv

load_dotenv(find_dotenv(usecwd=True), override=True)
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")
print("API Key Loaded:", os.getenv("GROQ_API_KEY")[:10] if os.getenv("GROQ_API_KEY") else "FAILED TO LOAD")




API Key Loaded: gsk_8yEZ0T


In [16]:
#Test the API KEY
from langchain_groq import ChatGroq
from langchain_openai import ChatOpenAI #alternative 

llm_model = ChatGroq(model ="qwen/qwen3.6-27b" )
print(llm_model.invoke("hello, tell me about yourself in few words").content)


<think>
Here's a thinking process:

1.  **Analyze User Input:**
   - **Greeting:** "hello"
   - **Request:** "tell me about yourself in few words"
   - **Key Constraints:** "in few words" (concise, brief)

2.  **Identify Core Identity/Role:**
   - I am Qwen, a large language model developed by Alibaba Group's Tongyi Lab.
   - My purpose is to assist with information, problem-solving, creativity, and conversation.
   - I'm designed to be helpful, accurate, and adaptable.

3.  **Draft Response (Mental Refinement):**
   I'm Qwen, an AI assistant developed by Alibaba Group's Tongyi Lab. I'm here to help with questions, problem-solving, creativity, and everyday tasks—quickly, accurately, and conversationally.

4.  **Check Constraints:**
   - "Few words"? Yes, it's concise (around 20 words).
   - Covers identity, developer, purpose, and tone.
   - Matches the requested brevity.

5.  **Final Polish:**
   "I’m Qwen, an AI assistant developed by Alibaba Group’s Tongyi Lab. I’m here to help wit

In [17]:
from langchain.tools import tool

@tool
def get_weather(location:str)->str:
    """ Get the weather at a location"""
    return f"its sunny in {location}"

model_with_tool = llm_model.bind_tools([get_weather])

#alternative 
# agent= create_agent(
# model = llm_model,tools = [get_weather], system_prompt ="You are a AI assistance")


In [18]:
response = model_with_tool.invoke("What is the weather like in boston")
print(response)
for tool_call in response.tool_calls:
    print(f"tool:{ tool_call['name']}")
    print(f"Args: {tool_call['args']}")

content='' additional_kwargs={'reasoning_content': 'Thinking Process:\n1.  Identify the user\'s request: Weather in Boston.\n2.  Identify the available tool: `get_weather` with parameter `location`.\n3.  Extract the location: "boston".\n4.  Call the tool: `get_weather(location="boston")`.\n5.  Formulate the response based on the tool\'s output. (Wait for tool output) -> I will simulate the tool call now.\n\nLet\'s call the function.✅\n', 'tool_calls': [{'id': 'sja2csts7', 'function': {'arguments': '{"location":"boston"}', 'name': 'get_weather'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 132, 'prompt_tokens': 276, 'total_tokens': 408, 'completion_time': 0.267001059, 'completion_tokens_details': {'reasoning_tokens': 103}, 'prompt_time': 0.021495839, 'prompt_tokens_details': None, 'queue_time': 0.049434384, 'total_time': 0.288496898}, 'model_name': 'qwen/qwen3.6-27b', 'system_fingerprint': 'fp_fff3b79855', 'service_tier': 'on_demand', 'finish_reason': '

In [19]:
print(response.tool_calls)

[{'name': 'get_weather', 'args': {'location': 'boston'}, 'id': 'sja2csts7', 'type': 'tool_call'}]


In [20]:
#Tool Execution Loops

messages = [{"role":"user","content":"What the weather is boston"}]
ai_msg = model_with_tool.invoke(messages)
print(ai_msg)
messages.append(ai_msg)
print(messages)

#execute the tool and collect the result 
for tool_call in ai_msg.tool_calls:
    tool_result = get_weather.invoke(tool_call)
    messages.append(tool_result)

#step3: pass results back to model fro final response
final_response = model_with_tool.invoke(messages)
print(final_response.text)


content='' additional_kwargs={'reasoning_content': 'Here\'s a thinking process:\n\n1.  **Understand User Input:** The user is asking "What the weather is boston" (likely meaning "What is the weather in Boston?").\n2.  **Identify Intent:** The user wants to know the current weather conditions in Boston.\n3.  **Check Available Tools:** I have a tool named `get_weather` that takes a `location` parameter (string).\n4.  **Map Input to Tool:** The location is "Boston". I need to call `get_weather` with `location: "Boston"`.\n5.  **Execute Tool Call:** Generate the function call.\n6.  **Handle Response:** (I will simulate the tool call and then process the result when it comes back, but for now, I just output the tool call.)\n\nLet\'s make the tool call.✅\n', 'tool_calls': [{'id': 'zjpthvsgb', 'function': {'arguments': '{"location":"Boston"}', 'name': 'get_weather'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 209, 'prompt_tokens': 274, 'total_tokens': 483, '

In [21]:
#Tool Message

from langchain.messages import SystemMessage,HumanMessage,AIMessage

messages = [
    SystemMessage("You are writer"),
    HumanMessage("Write a short article on Ai")
]
response = llm_model.invoke(messages)
response.content

'\n<think>\nHere\'s a thinking process:\n\n1.  **Analyze User Input:**\n   - **Topic:** AI (Artificial Intelligence)\n   - **Format:** Short article\n   - **Role:** Writer (implicit in the prompt)\n   - **Key Requirements:** Informative, engaging, well-structured, appropriate length for a "short article"\n\n2.  **Determine Article Structure:**\n   A standard short article structure:\n   - Catchy Title\n   - Engaging Introduction (hook + thesis/overview)\n   - Body Paragraph 1: What is AI? (Definition & Evolution)\n   - Body Paragraph 2: How it\'s transforming industries (Applications)\n   - Body Paragraph 3: Challenges & Ethical Considerations\n   - Conclusion: Future outlook + final thought\n   - Keep it concise (approx. 500-700 words)\n\n3.  **Draft - Section by Section:**\n\n   *Title Ideas:*\n   - The Age of Intelligence: How AI is Reshaping Our World\n   - Beyond the Hype: Understanding Artificial Intelligence\n   - Artificial Intelligence: The Quiet Revolution\n   *Selected:* The

In [22]:

from langchain.messages import AIMessage, SystemMessage, HumanMessage,AIMessage

#create AI message mually (eg for conversation history)
ai_msg = AIMessage("Id be happy to help you with that question")
messages= [
    SystemMessage("You are a writer"),
    HumanMessage("Write a short article on Ai"),
    ai_msg,
    HumanMessage("Great! what is 2+2")
]

response = llm_model.invoke(messages)
response.content

'\n<think>\nHere\'s a thinking process:\n\n1.  **Analyze User Input:**\n   - User said: "Great! what is 2+2"\n   - This is a simple math question, but it follows a previous request where I acknowledged I\'d help write a short article on AI.\n   - The user seems to be testing me or just asking a straightforward question.\n\n2.  **Identify Core Task:**\n   - Answer the math question: 2 + 2 = 4\n   - Maintain a helpful, conversational tone\n   - Optionally, bridge back to the original request (AI article) if appropriate, but keep it concise.\n\n3.  **Formulate Response:**\n   - Direct answer: 2 + 2 = 4\n   - Keep it simple and accurate\n   - Offer to continue with the AI article if they\'re ready\n\n   Draft: "2 + 2 = 4! If you\'re ready, I’d be happy to write that short article on AI for you. Just let me know your preferred tone, length, or any specific focus areas you’d like covered."\n\n4.  **Self-Correction/Refinement:**\n   - The response is accurate, friendly, and stays on track.\n 

In [23]:
print(response )    

content='\n<think>\nHere\'s a thinking process:\n\n1.  **Analyze User Input:**\n   - User said: "Great! what is 2+2"\n   - This is a simple math question, but it follows a previous request where I acknowledged I\'d help write a short article on AI.\n   - The user seems to be testing me or just asking a straightforward question.\n\n2.  **Identify Core Task:**\n   - Answer the math question: 2 + 2 = 4\n   - Maintain a helpful, conversational tone\n   - Optionally, bridge back to the original request (AI article) if appropriate, but keep it concise.\n\n3.  **Formulate Response:**\n   - Direct answer: 2 + 2 = 4\n   - Keep it simple and accurate\n   - Offer to continue with the AI article if they\'re ready\n\n   Draft: "2 + 2 = 4! If you\'re ready, I’d be happy to write that short article on AI for you. Just let me know your preferred tone, length, or any specific focus areas you’d like covered."\n\n4.  **Self-Correction/Refinement:**\n   - The response is accurate, friendly, and stays on t

PYDANTIC: provides the richest feature set with field validation, descriptions 

In [36]:
#using BaseModel

from pydantic import BaseModel, Field

class Movie(BaseModel):
    title:str = Field(description = " The title of the movie")
    year:int = Field(description = "This is the year the movie was released as an integer, example: 1990")
    director:str = Field(description="The director of the movie")
    rating:int = Field(description=" The movie ratings out of 10 as an integer, example: 8,9,2")



In [37]:
model_with_structure = llm_model.with_structured_output(Movie)
model_with_structure

_ChatModelBinding(bound=ChatGroq(metadata={'lc_versions': {'langchain-core': '1.5.4', 'langchain': '1.3.15'}}, client=<groq.resources.chat.completions.Completions object at 0x0000020FA2769EE0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x0000020FA2768B00>, model_name='qwen/qwen3.6-27b', model_kwargs={}, groq_api_key=SecretStr('**********')), kwargs={'tools': [{'type': 'function', 'function': {'name': 'Movie', 'description': '', 'parameters': {'properties': {'title': {'description': ' The title of the movie', 'type': 'string'}, 'year': {'description': 'This is the year the movie was released as an integer, example: 1990', 'type': 'integer'}, 'director': {'description': 'The director of the movie', 'type': 'string'}, 'rating': {'description': ' The movie ratings out of 10 as an integer, example: 8,9,2', 'type': 'integer'}}, 'required': ['title', 'year', 'director', 'rating'], 'type': 'object'}}}], 'ls_structured_output_format': {'kwargs': {'method': 'functi

In [38]:
response = model_with_structure.invoke("Provide the details about the movie jurasic")
print(response)

title='Jurassic Park' year=1993 director='Steven Spielberg' rating=8


In [39]:
model_with_structure = llm_model.with_structured_output(Movie, include_raw=True)

response = model_with_structure.invoke("PRovide details about the movie Inception ")
response 

{'raw': AIMessage(content='', additional_kwargs={'reasoning_content': 'Here\'s a thinking process:\n\n1.  **Analyze User Input:**\n   - User asks: "PRovide details about the movie Inception"\n   - Key entity: Movie "Inception"\n   - Required information from the tool: title, year, director, rating\n\n2.  **Identify Tool Requirements:**\n   - Tool: `Movie`\n   - Parameters: `title` (string), `year` (integer), `director` (string), `rating` (integer, out of 10)\n   - All parameters are required.\n\n3.  **Gather Information about "Inception":**\n   - Title: Inception\n   - Year: 2010\n   - Director: Christopher Nolan\n   - Rating: I need to provide a rating out of 10 as an integer. Common ratings for Inception: IMDB ~8.8, Rotten Tomatoes ~87%. I\'ll use 9 as a reasonable integer rating out of 10, or I could stick to a well-known rating like 8 or 9. The prompt says "ratings out of 10 as an integer, example: 8,9,2". I\'ll use 9.\n\n   Let\'s verify facts:\n   - Title: Inception\n   - Year: 2

In [40]:
#Nest Structure

from pydantic import BaseModel, Field
class Actor(BaseModel):
    name:str
    role:str
class MovieDetails(BaseModel):
    title:str
    year:int
    cast:list[Actor]
    genres:list[str]
    budget:float | None = Field(None, description="budget is millions USD")

model_with_structure = llm_model.with_structured_output(MovieDetails)
respone = model_with_structure.invoke("provide detials about the movie jurrasic")
response

{'raw': AIMessage(content='', additional_kwargs={'reasoning_content': 'Here\'s a thinking process:\n\n1.  **Analyze User Input:**\n   - User asks: "PRovide details about the movie Inception"\n   - Key entity: Movie "Inception"\n   - Required information from the tool: title, year, director, rating\n\n2.  **Identify Tool Requirements:**\n   - Tool: `Movie`\n   - Parameters: `title` (string), `year` (integer), `director` (string), `rating` (integer, out of 10)\n   - All parameters are required.\n\n3.  **Gather Information about "Inception":**\n   - Title: Inception\n   - Year: 2010\n   - Director: Christopher Nolan\n   - Rating: I need to provide a rating out of 10 as an integer. Common ratings for Inception: IMDB ~8.8, Rotten Tomatoes ~87%. I\'ll use 9 as a reasonable integer rating out of 10, or I could stick to a well-known rating like 8 or 9. The prompt says "ratings out of 10 as an integer, example: 8,9,2". I\'ll use 9.\n\n   Let\'s verify facts:\n   - Title: Inception\n   - Year: 2

## TYPED DICT:
provides a simpler alternative using pythong built in typing, ideal when you dont need runtime validation 

In [47]:
from typing_extensions import TypedDict, Annotated

from langchain_core.messages import Annotation
class MovieDict(TypedDict):
    """ A Movie with details """
    title: Annotated[str,...," the title of the mvoie"]
    year: Annotated[int,...,"the year the movie was released"]
    director:Annotated[str,...,"the director of hte movie"]
    rating: Annotated[float,...," the movie's rating out of 10"]


model_with_typeddict = llm_model.with_structured_output(MovieDict)
response = model_with_typeddict.invoke("Please provide the details of the movie avengers")
response 

{'director': 'Joss Whedon', 'rating': 8, 'title': 'The Avengers', 'year': 2012}

In [48]:
#NESTED Structure 
#pydantic: BaseModel

class Actor(TypedDict):
    name:str
    role:str
class MovieDetails(TypedDict):
    title: str
    year: int
    cast: list[Actor]
    genres: list[str]
    budget:float | None = Field(None, descrption = "budget is millions USD as in float, example 10.0")

model_with_structure = llm_model.with_structured_output(MovieDetails)

response = model_with_structure.invoke("provide details about the movie jurrasic")
response
#error if we remove... as in float,example 10.0


C:\Users\rupak\AppData\Local\Temp\ipykernel_13500\1452260950.py:12: PydanticDeprecatedSince20: Using extra keyword arguments on `Field` is deprecated and will be removed. Use `json_schema_extra` instead. (Extra keys: 'descrption'). Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.13/migration/
  budget:float | None = Field(None, descrption = "budget is millions USD as in float, example 10.0")


{'budget': 63000000,
 'cast': [{'name': 'Sam Neill', 'role': 'Dr. Alan Grant'},
  {'name': 'Laura Dern', 'role': 'Dr. Ellie Sattler'},
  {'name': 'Jeff Goldblum', 'role': 'Dr. Ian Malcolm'},
  {'name': 'Richard Attenborough', 'role': 'John Hammond'},
  {'name': 'Bob Peck', 'role': 'Robert Muldoon'},
  {'name': 'Martin Ferrero', 'role': 'Donald Gennaro'},
  {'name': 'BD Wong', 'role': 'Dr. Henry Wu'},
  {'name': 'Samuel L. Jackson', 'role': 'Ray Arnold'}],
 'genres': ['Action', 'Adventure', 'Science Fiction', 'Thriller'],
 'title': 'Jurassic Park',
 'year': 1993}

# Integrated with the Agent 
helps to reduce token cost A2A communication

In [50]:
#integrate it with the agent

from langchain.agents import create_agent

class ContactInfo(BaseModel):
    """Contact information for a person"""
    name:str = Field(description = "The name of the  person")
    email:str = Field(description="The email address of the person")
    phone:int = Field(description="the phone number of the person as in integer, example: 2232334")

agent = create_agent(
model = llm_model,
response_format = ContactInfo #aselects provded Strategy
)

result = agent.invoke({
    "messages":[{"role":"user","content":"Extract the contact information from bob, bob@gmail.com, phone numer: 334456,"}]
})

print(result["structured_response"])

name='bob' email='bob@gmail.com' phone=334456


In [ ]:
#using TypeDict 
#integrate it with the Agent

from typing_extensions import TypedDict
from langchain.agents import create_agent

class ContactInfo(TypedDict):
    """Contact information for a person """
    name: str # the name of the person 
    email: str #the email address of the person 
    phone: str  #the phone number of the person

agent = create_agent(
model = llm_model, 
        response_format = ContactInfo #selects providerStrategy

)
result = agent.invoke({
    "messages":[{"role":"user","content": "Extract contact info from: John Doe,John@example.com, (555) 123-4567"}]
})
print(result["structured_response"])

{'name': 'John Doe', 'email': 'John@example.com', 'phone': '(555) 123-4567'}


### DATA CLASS
Building standard internal Python objects/classes where you want basic object attributes (obj.attr) without needing Pydantic's validation overhead or third-party packages.

In [ ]:
### Data Class

from dataclasses import dataclass
from langchain.agents import create_agent

@dataclass
class ContactInfo:
    """ contcat information for a person"""
    name: str # the name of the person
    email: str # the email address of the person
    phone: str # the phone number of th eperson

agent = create_agent(
    model = llm_model, response_format = ContactInfo
)

result = agent.invoke({
    "messages":[{"role":"user","content": "Extract contact info from: John Doe,John@example.com, (555) 123-4567"}]
})
print(result["structured_response"])

ContactInfo(name='John Doe', email='John@example.com', phone='(555) 123-4567')
